In [39]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import matplotlib.pyplot as plt


ALPHA = 0.05

# Load file
df = pd.read_csv("../results_analysis_1/vnd_results.csv")
df = pd.read_csv("vns_results.csv")

df

,instance,order,best_known,cost,delta_percent,time_seconds
0,N-tiw56r66_150,order1,1940755,1914923,1.331028,7.019231
1,N-tiw56r66_150,order2,1940755,1892867,2.467493,12.010030
2,N-stabu1_150,order1,2875732,2840013,1.242084,13.680416
3,N-stabu1_150,order2,2875732,2847149,0.993938,29.079103
4,N-tiw56r67_150,order1,2056347,2037509,0.916091,22.849983
...,...,...,...,...,...,...
73,N-t59b11xx_150,order2,3239550,3195780,1.351114,18.431845
74,N-t59d11xx_150,order1,1462697,1450165,0.856773,6.618589
75,N-t59d11xx_150,order2,1462697,1446348,1.117730,8.309361
76,N-t75e11xx_150,order1,41570193,41214145,0.856498,12.367771


In [40]:
# Fastest order 

quality = df.pivot(index="instance", columns="order", values="delta_percent").reset_index()
quality.columns.name = None
quality = quality.dropna(subset=["order1", "order2"]).copy()

xq = quality["order1"].to_numpy()
yq = quality["order2"].to_numpy()
diff_q = xq - yq   # negative => order1 better

mean_q1 = np.mean(xq)
mean_q2 = np.mean(yq)

tq_stat, tq_p = ttest_rel(xq, yq)
if np.allclose(diff_q, 0):
    wq_stat, wq_p = np.nan, 1.0
else:
    wq_stat, wq_p = wilcoxon(xq, yq, zero_method="wilcox", alternative="two-sided")

best_quality_order = "order1" if mean_q1 < mean_q2 else ("order2" if mean_q2 < mean_q1 else "tie")


print("=== VND: Solution quality ===")
print(f"Mean delta order1: {mean_q1:.6f}")
print(f"Mean delta order2: {mean_q2:.6f}")
print(f"Best quality order: {best_quality_order}")

=== VND: Solution quality ===
Mean delta order1: 1.187995
Mean delta order2: 1.315720
Best quality order: order1


In [ ]:
# ----- Time pairing -----
time_df = df.pivot(index="instance", columns="order", values="time_seconds").reset_index()
time_df.columns.name = None
time_df = time_df.dropna(subset=["order1", "order2"]).copy()

xt = time_df["order1"].to_numpy()
yt = time_df["order2"].to_numpy()
diff_t = xt - yt   # negative => order1 faster

mean_t1 = np.mean(xt)
mean_t2 = np.mean(yt)

tt_stat, tt_p = ttest_rel(xt, yt)
if np.allclose(diff_t, 0):
    wt_stat, wt_p = np.nan, 1.0
else:
    wt_stat, wt_p = wilcoxon(xt, yt, zero_method="wilcox", alternative="two-sided")

fastest_order = "order1" if mean_t1 < mean_t2 else ("order2" if mean_t2 < mean_t1 else "tie")

print("=== VND: Computation time ===")
print(f"Mean time order1: {mean_t1:.6f} s")
print(f"Mean time order2: {mean_t2:.6f} s")
print(f"Sum time order1 : {sum_t1:.6f} s")
print(f"Sum time order2 : {sum_t2:.6f} s")
print(f"Fastest order: {fastest_order}")

=== VND: Computation time ===
Mean time order1: 11.766360 s
Mean time order2: 18.825943 s
Sum time order1 : 458.888055 s
Sum time order2 : 734.211783 s
Fastest order: order1


In [42]:
# ------------------------------------------------------------
# Per-instance comparison table
# ------------------------------------------------------------
quality = df.pivot(index="instance", columns="order", values="delta_percent").reset_index()
quality.columns.name = None

time_tbl = df.pivot(index="instance", columns="order", values="time_seconds").reset_index()
time_tbl.columns.name = None

comparison_table = quality.merge(time_tbl, on="instance", suffixes=("_delta", "_time"))
comparison_table = comparison_table.rename(columns={
    "order1_delta": "order1_delta_percent",
    "order2_delta": "order2_delta_percent",
    "order1_time": "order1_time_seconds",
    "order2_time": "order2_time_seconds",
})

comparison_table["better_quality"] = comparison_table.apply(
    lambda row: "order1" if row["order1_delta_percent"] < row["order2_delta_percent"]
    else ("order2" if row["order2_delta_percent"] < row["order1_delta_percent"] else "tie"),
    axis=1
)

comparison_table["faster_order"] = comparison_table.apply(
    lambda row: "order1" if row["order1_time_seconds"] < row["order2_time_seconds"]
    else ("order2" if row["order2_time_seconds"] < row["order1_time_seconds"] else "tie"),
    axis=1
)

comparison_table["diff_delta_order1_minus_order2"] = (
    comparison_table["order1_delta_percent"] - comparison_table["order2_delta_percent"]
)
comparison_table["diff_time_order1_minus_order2"] = (
    comparison_table["order1_time_seconds"] - comparison_table["order2_time_seconds"]
)
comparison_table

,instance,order1_delta_percent,order2_delta_percent,order1_time_seconds,order2_time_seconds,better_quality,faster_order,diff_delta_order1_minus_order2,diff_time_order1_minus_order2
0,N-be75eec_150,0.273542,0.928527,12.857323,15.189174,order1,order1,-0.654985,-2.331851
1,N-be75np_150,0.719399,1.394514,20.330251,17.095889,order1,order2,-0.675115,3.234362
2,N-be75oi_150,0.503932,0.710561,21.923932,14.541273,order1,order2,-0.206629,7.382659
3,N-be75tot_150,1.438983,1.889409,6.771444,11.701983,order1,order1,-0.450426,-4.930539
4,N-stabu1_150,1.242084,0.993938,13.680416,29.079103,order2,order1,0.248146,-15.398687
5,N-stabu2_150,0.808381,1.507092,17.514002,26.949913,order1,order1,-0.698711,-9.435911
6,N-stabu3_150,0.744317,0.817369,12.613217,13.628904,order1,order1,-0.073052,-1.015687
7,N-t59b11xx_150,1.303576,1.351114,8.652870,18.431845,order1,order1,-0.047538,-9.778975
8,N-t59d11xx_150,0.856773,1.117730,6.618589,8.309361,order1,order1,-0.260957,-1.690772
9,N-t59f11xx_150,1.412550,1.358979,8.008059,19.005491,order2,order1,0.053571,-10.997432


In [43]:
paired = df.pivot(index="instance", columns="order", values="delta_percent").reset_index()

paired 

order,instance,order1,order2
0,N-be75eec_150,0.273542,0.928527
1,N-be75np_150,0.719399,1.394514
2,N-be75oi_150,0.503932,0.710561
3,N-be75tot_150,1.438983,1.889409
4,N-stabu1_150,1.242084,0.993938
5,N-stabu2_150,0.808381,1.507092
6,N-stabu3_150,0.744317,0.817369
7,N-t59b11xx_150,1.303576,1.351114
8,N-t59d11xx_150,0.856773,1.117730
9,N-t59f11xx_150,1.412550,1.358979


In [44]:
# Extract paired samples
x = paired["order1"].to_numpy()
y = paired["order2"].to_numpy()

# Descriptive statistics
mean_order1 = np.mean(x)
mean_order2 = np.mean(y)
diff = x - y   # negative => order1 better, positive => order2 better

print("\n=== Descriptive statistics ===")
print(f"Number of common instances: {len(paired)}")
print(f"Mean delta order1: {mean_order1:.6f}")
print(f"Mean delta order2: {mean_order2:.6f}")
print(f"Mean difference (order1 - order2): {np.mean(diff):.6f}")
print(f"Median difference (order1 - order2): {np.median(diff):.6f}")

if mean_order1 < mean_order2:
    print("Better average solution quality: order1")
elif mean_order2 < mean_order1:
    print("Better average solution quality: order2")
else:
    print("Same average solution quality")

# Paired t-test
t_stat, t_pvalue = ttest_rel(x, y)

# Wilcoxon signed-rank test
if np.allclose(diff, 0):
    w_stat = np.nan
    w_pvalue = 1.0
else:
    w_stat, w_pvalue = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")

print("\n=== Statistical hypothesis tests ===")
print(f"Alpha = {ALPHA}")

print("\nPaired t-test")
print(f"t-statistic = {t_stat:.6f}")
print(f"p-value     = {t_pvalue:.6f}")
if t_pvalue < ALPHA:
    print("Reject H0: significant difference between order1 and order2")
else:
    print("Do not reject H0: no statistically significant difference")

print("\nWilcoxon signed-rank test")
print(f"statistic = {w_stat}")
print(f"p-value   = {w_pvalue:.6f}")
if w_pvalue < ALPHA:
    print("Reject H0: significant difference between order1 and order2")
else:
    print("Do not reject H0: no statistically significant difference")

#If you want a compact sentence for the report, use this too:

if mean_order1 < mean_order2:
    better = "order1"
else:
    better = "order2"

print("\n=== Report sentence ===")
print(
    f"On the common instances, {better} achieves the lower average relative deviation. "
    f"The paired t-test gives p = {t_pvalue:.6f} and the Wilcoxon signed-rank test gives "
    f"p = {w_pvalue:.6f}. At alpha = {ALPHA}, "
    f"{'the difference is statistically significant' if (t_pvalue < ALPHA or w_pvalue < ALPHA) else 'the difference is not statistically significant'}."
)



=== Descriptive statistics ===
Number of common instances: 39
Mean delta order1: 1.187995
Mean delta order2: 1.315720
Mean difference (order1 - order2): -0.127725
Median difference (order1 - order2): -0.024351
Better average solution quality: order1

=== Statistical hypothesis tests ===
Alpha = 0.05

Paired t-test
t-statistic = -1.492156
p-value     = 0.143915
Do not reject H0: no statistically significant difference

Wilcoxon signed-rank test
statistic = 309.0
p-value   = 0.264514
Do not reject H0: no statistically significant difference

=== Report sentence ===
On the common instances, order1 achieves the lower average relative deviation. The paired t-test gives p = 0.143915 and the Wilcoxon signed-rank test gives p = 0.264514. At alpha = 0.05, the difference is not statistically significant.
